In [72]:
import pydantic as pyd
pyd.__version__

'2.13.4'

### Basic Pydantic Example

In [ ]:
from pydantic import BaseModel

In [12]:
class employee(BaseModel):
    name: str
    age: int
    department: str

In [16]:
obj1 = employee(name="John Doe", age="129", department="HR")
print(obj1)

name='John Doe' age=129 department='HR'


### StrictType and Fields Use

In [48]:
# creating a hard validation on the data, there is noparsing involved
from pydantic import Field, StrictInt, StrictStr, StrictFloat, EmailStr
from typing import Optional

class employee(BaseModel):
    name: StrictStr = Field(..., min_length=2, max_length=50)
    age: StrictInt = Field(..., gt=18, lt=63)
    department: StrictStr = Field()
    salary: Optional[StrictFloat] = Field(..., gt=1200000)
    email: EmailStr = Field(..., description="Employee's Email Address")

data1 = {'name': "Hiten",
        "age": 21,
        "department": "Research Engineer",
        "email": "hjain9261@gmail.com",
        "salary": 10000000
}

obj1 = employee(**data1)
type(obj1)

__main__.employee

### Dumping
The process of converting a structured Pydantic model instance into a less structured format, such as a Python dictionary or a JSON-encoded string

In [52]:
dict_obj1 = obj1.model_dump() ## Creates A Dictionary 
json_obj1 = obj1.model_dump_json() ## Creates A JSON

dict_obj1, json_obj1

({'name': 'Hiten',
  'age': 21,
  'department': 'Research Engineer',
  'salary': 10000000.0,
  'email': 'hjain9261@gmail.com'},
 '{"name":"Hiten","age":21,"department":"Research Engineer","salary":10000000.0,"email":"hjain9261@gmail.com"}')

### Model Example: ReaL World

In [53]:
from datetime import datetime
from pydantic import BaseModel, Field, EmailStr, PositiveFloat, PositiveInt
from typing import List, Optional

class ProductItem(BaseModel):
    name: str
    price: PositiveFloat
    quantity: PositiveInt

class Order(BaseModel):
    order_id: int = Field(description = "Unique business Database Identifier")
    customer_email: EmailStr 
    items: List[ProductItem]   # Nested Model
    discount: Optional[PositiveFloat] = 0.0
    created_at: datetime

In [70]:
# valid data 
data = {
    "order_id": "6901",
    "customer_email": "hjain9261@gmail.com",
    "items": [
       {"name": "Developer Mechanical Keyboard", "price": 150.00, "quantity": 1},
        {"name": "USB-C Hub", "price": "45.50", "quantity": 2} 
    ],
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

obj = Order(**data).model_dump()
obj  # Auto Parsed the int -> str

{'order_id': 6901,
 'customer_email': 'hjain9261@gmail.com',
 'items': [{'name': 'Developer Mechanical Keyboard',
   'price': 150.0,
   'quantity': 1},
  {'name': 'USB-C Hub', 'price': 45.5, 'quantity': 2}],
 'discount': 0.0,
 'created_at': datetime.datetime(2026, 6, 18, 7, 9, 43)}

In [71]:
## Error handling for corrupted_data

from pydantic import ValidationError

corrupted_data = {
    "order_id": "NaN",
    "customer_email": "hi234@example",
    "items": [],
    "created_at": "2026_06_16"
}

try:
    Order(**corrupted_data)
except ValidationError as ve:
    print(ve.json(indent=2))

[
  {
    "type": "int_parsing",
    "loc": [
      "order_id"
    ],
    "msg": "Input should be a valid integer, unable to parse string as an integer",
    "input": "NaN",
    "url": "https://errors.pydantic.dev/2.13/v/int_parsing"
  },
  {
    "type": "value_error",
    "loc": [
      "customer_email"
    ],
    "msg": "value is not a valid email address: The part after the @-sign is not valid. It should have a period.",
    "input": "hi234@example",
    "ctx": {
      "reason": "The part after the @-sign is not valid. It should have a period."
    },
    "url": "https://errors.pydantic.dev/2.13/v/value_error"
  },
  {
    "type": "datetime_from_date_parsing",
    "loc": [
      "created_at"
    ],
    "msg": "Input should be a valid datetime or date, invalid date separator, expected `-`",
    "input": "2026_06_16",
    "ctx": {
      "error": "invalid date separator, expected `-`"
    },
    "url": "https://errors.pydantic.dev/2.13/v/datetime_from_date_parsing"
  }
]
